# SQL query from table names - Continued

In [1]:
from openai import OpenAI
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

## The old Prompt

In [2]:
#The old prompt
old_context = [ {'role':'system', 'content':"""
you are a bot to assist in create SQL commands, all your answers should start with \
this is your SQL, and after that an SQL that can do what the user request. \
Your Database is composed by a SQL database with some tables. \
Try to maintain the SQL order simple.
Put the SQL command in white letters with a black background, and just after \
a simple and concise text explaining how it works.
If the user ask for something that can not be solved with an SQL Order \
just answer something nice and simple, maximum 10 words, asking him for something that \
can be solved with SQL.
"""} ]

old_context.append( {'role':'system', 'content':"""
first table:
{
  "tableName": "employees",
  "fields": [
    {
      "nombre": "ID_usr",
      "tipo": "int"
    },
    {
      "nombre": "name",
      "tipo": "varchar"
    }
  ]
}
"""
})

old_context.append( {'role':'system', 'content':"""
second table:
{
  "tableName": "salary",
  "fields": [
    {
      "nombre": "ID_usr",
      "type": "int"
    },
    {
      "name": "year",
      "type": "date"
    },
    {
      "name": "salary",
      "type": "float"
    }
  ]
}
"""
})

old_context.append( {'role':'system', 'content':"""
third table:
{
  "tablename": "studies",
  "fields": [
    {
      "name": "ID",
      "type": "int"
    },
    {
      "name": "ID_usr",
      "type": "int"
    },
    {
      "name": "educational_level",
      "type": "int"
    },
    {
      "name": "Institution",
      "type": "varchar"
    },
    {
      "name": "Years",
      "type": "date"
    }
    {
      "name": "Speciality",
      "type": "varchar"
    }
  ]
}
"""
})

## New Prompt.
We are going to improve it following the instructions of a Paper from the Ohaio University: [How to Prompt LLMs for Text-to-SQL: A Study in Zero-shot, Single-domain, and Cross-domain Settings](https://arxiv.org/abs/2305.11853). I recommend you read that paper.

For each table, we will define the structure using the same syntax as in a SQL create table command, and add the sample rows of the content.

Finally, at the end of the prompt, we'll include some example queries with the SQL that the model should generate. This technique is called Few-Shot Samples, in which we provide the prompt with some examples to assist it in generating the correct SQL.


In [31]:
context = [ {'role':'system', 'content':"""

CREATE TABLE employees (
            ID_usr INTEGER,     -- unique identifier for each employee 
            name VARCHAR(80)    -- full name of the employee
            );
             
CREATE TABLE salary (
            ID_usr INTEGER,     -- reference to employees.id
            year DATE,          -- the year for which the salary is recorded
            salary FLOAT        -- annual salary amount
            );

CREATE TABLE studies (
            ID_usr INTEGER,             -- reference to employees.id 
            educational_level VARCHAR(80), -- academic level (e.g. MSc, MBA)
            Institution VARCHAR(80),    -- name of the educational institution
            Years DATE,                 -- number of years to finish the study 
            Speciality VARCHAR(100)     -- field of study (e.g., Computer Science)
            );                    
"""} ]



In [32]:
#FEW SHOT SAMPLES
context.append( {'role':'system', 'content':"""
 -- Maintain the SQL order simple and efficient as you can, using valid SQL Lite, answer the following questions for the table provided above.
SELECT name FROM employees WHERE ID_usr = 100;
SELECT SUM(*) FROM salary;
"""
})

In [33]:
#Functio to call the model.
def return_CCRMSQL(user_message, context):
    client = OpenAI(
    # This is the default and can be omitted
    api_key=OPENAI_API_KEY,
)

    newcontext = context.copy()
    newcontext.append({'role':'user', 'content':"question: " + user_message})

    response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=newcontext,
            temperature=0,
        )

    return (response.choices[0].message.content)

## NL2SQL Samples
We're going to review some examples generated with the old prompt and others with the new prompt.

In [34]:
#new
context_user = context.copy()
print(return_CCRMSQL("""List the employee with the highest salary.""", context_user))

```sql
SELECT e.name
FROM employees e
JOIN salary s ON e.ID_usr = s.ID_usr
WHERE s.salary = (SELECT MAX(salary) FROM salary);
```


In [35]:
#old
old_context_user = old_context.copy()
print(return_CCRMSQL("List the employee with the highest salary.", old_context_user))

This is your SQL:
```sql
SELECT e.name, s.salary
FROM employees e
JOIN salary s ON e.ID_usr = s.ID_usr
ORDER BY s.salary DESC
LIMIT 1;
```

This SQL query selects the name and salary of employees by joining the "employees" and "salary" tables on the ID_usr column. It then orders the results by salary in descending order and limits the output to only the first row, which represents the employee with the highest salary.


In [36]:
#new
print(return_CCRMSQL("Display top 1 highest average salary of employees grouped by their corresponding study institution.", context_user))

```sql
SELECT s.Institution, AVG(sa.salary) AS avg_salary
FROM employees e
JOIN salary sa ON e.ID_usr = sa.ID_usr
JOIN studies s ON e.ID_usr = s.ID_usr
GROUP BY s.Institution
ORDER BY avg_salary DESC
LIMIT 1;
```


In [37]:
#old
print(return_CCRMSQL("Display top 1 highest average salary of employees grouped by their corresponding study institution.", old_context_user))

This is your SQL:
```sql
SELECT e.name, s.Institution, AVG(s.salary) AS avg_salary
FROM employees e
JOIN salary s ON e.ID_usr = s.ID_usr
GROUP BY s.Institution
ORDER BY avg_salary DESC
LIMIT 1;
```

This SQL query retrieves the name of the employee, the institution they studied at, and the average salary of employees grouped by their corresponding study institution. It then orders the results by the average salary in descending order and limits the output to display only the top 1 highest average salary.


# Exercise
 - Complete the prompts similar to what we did in class. 
     - Try at least 3 versions
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong.
     - What did you learn?

In [38]:
#new
print(return_CCRMSQL("Group the employees based on their specialities.", context_user))

To group the employees based on their specialities, you can use the following SQL query:

```sql
SELECT e.name, s.Speciality
FROM employees e
JOIN studies s ON e.ID_usr = s.ID_usr
ORDER BY s.Speciality;
``` 

This query will retrieve the names of employees along with their specialities, grouping them based on their field of study.


In [39]:
#new
print(return_CCRMSQL("Group the employees based on their specialities.", old_context_user))

This is your SQL:
```sql
SELECT Speciality, COUNT(*) as TotalEmployees
FROM studies
GROUP BY Speciality;
```

This SQL query selects the speciality field from the studies table and counts the number of employees for each speciality by grouping the results based on the speciality column.


In [40]:
#new
print(return_CCRMSQL("Display the person that has worked for the most years in the company.", context_user))

```sql
SELECT e.name
FROM employees e
JOIN (
    SELECT ID_usr, MAX(Years) AS max_years
    FROM studies
    GROUP BY ID_usr
) s ON e.ID_usr = s.ID_usr
WHERE s.max_years = (
    SELECT MAX(Years)
    FROM studies
);
```


In [41]:
#old
print(return_CCRMSQL("Display the person that has worked for the most years in the company.", old_context_user))

This is your SQL:
```sql
SELECT e.name, MAX(salary.year - studies.Years) AS total_work_years
FROM employees e
JOIN salary ON e.ID_usr = salary.ID_usr
JOIN studies ON e.ID_usr = studies.ID_usr
GROUP BY e.name
ORDER BY total_work_years DESC
LIMIT 1;
```

This SQL query joins the three tables (employees, salary, and studies) on the employee ID. It calculates the total work years by subtracting the year of the salary from the years of studies for each employee. Then, it selects the employee with the maximum total work years and displays their name.


In [42]:
#new
print(return_CCRMSQL("Display the employees with the highest academic degrees.", context_user))

```sql
SELECT e.name, s.educational_level
FROM employees e
JOIN studies s ON e.ID_usr = s.ID_usr
WHERE s.educational_level = (SELECT MAX(educational_level) FROM studies);
```


In [43]:
#old
print(return_CCRMSQL("Display the employees with the highest academic degrees.", old_context_user))

This is your SQL:
```sql
SELECT e.name, s.educational_level
FROM employees e
JOIN studies s ON e.ID_usr = s.ID_usr
ORDER BY s.educational_level DESC
LIMIT 1;
```

This SQL query retrieves the name of the employee and their highest academic degree by joining the "employees" and "studies" tables on the employee ID. It then orders the results by educational level in descending order and limits the output to only the top result, which represents the employee with the highest academic degree.


My observations:
- Fields from the tables are easily mixed, providing wrong SQL queries. With adaptations, the answer to the new question can generate proper expected result.
- For a query like "Display the person that has worked for the most years in the company." none of the models provides valid solutions.